# Connecting LangGraph with MCP
Companion notebook for the **Connecting LangGraph with MCP** section of Chapter 5.

Combines everything: an LLM in a LangGraph graph autonomously decides to call the MCP-hosted `search_web` / `search_wikipedia` tools (from notebook 03's server) to answer a multi-part query.

> **Run notebook 03 first** so `server.py` exists in this folder. Add `OPENAI_API_KEY` to your `.env` for a real run -- without it the agent still completes, it just skips the LLM call (same fallback pattern used throughout this book).

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Jupyter's stdout/stderr streams don't expose a real OS file descriptor, which crashes
# the MCP SDK's Windows subprocess-spawning fallback (it isn't needed outside Jupyter).
if os.name == "nt":
    try:
        sys.stderr.fileno()
    except Exception:
        sys.stderr = os.fdopen(os.dup(2), "w")

from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

load_dotenv()

False

In [2]:
async def make_agent_graph():
    print("Starting MCP Server and establishing connection...")
    server_path = Path.cwd() / "server.py"

    # MultiServerMCPClient can manage several MCP servers at once; here we only need one.
    client = MultiServerMCPClient({
        "search_server": {
            "transport": "stdio",
            "command": sys.executable,
            "args": [str(server_path), "stdio"],
            "env": os.environ.copy(),
        }
    })

    # Fetch the tools dynamically from the MCP server, in LangChain's tool format
    mcp_tools = await client.get_tools()
    print(f"Loaded {len(mcp_tools)} tools from MCP server.")

    # Falls back to a dummy key so the client can be constructed even when OPENAI_API_KEY
    # is not set; call_model() below stops it from ever being called for real.
    llm = ChatOpenAI(model="gpt-4o", temperature=0, api_key=os.getenv("OPENAI_API_KEY", "sk-not-set"))
    llm_with_tools = llm.bind_tools(mcp_tools)

    async def call_model(state: MessagesState):
        if not os.getenv("OPENAI_API_KEY"):
            return {"messages": [AIMessage(content="[Skipped LLM call: set OPENAI_API_KEY to run this agent for real]")]}
        response = await llm_with_tools.ainvoke(state["messages"])
        return {"messages": [response]}

    workflow = StateGraph(MessagesState)
    workflow.add_node("agent", call_model)
    workflow.add_node("tools", ToolNode(mcp_tools))  # ToolNode executes MCP tool calls natively

    workflow.add_edge(START, "agent")
    # tools_condition routes to "tools" if the LLM made a tool call, else straight to END
    workflow.add_conditional_edges("agent", tools_condition)
    workflow.add_edge("tools", "agent")

    app = workflow.compile()
    return app, client

In [3]:
app, client = await make_agent_graph()

print("\nGraph compiled. Sending query to the agent...\n")

query = (
    "Search the web for the latest news on the Model Context Protocol (MCP), "
    "then check Wikipedia for its creator or origins."
)

result = await app.ainvoke({"messages": [("user", query)]})

print("=== Final Response ===")
print(result["messages"][-1].content)

Starting MCP Server and establishing connection...


Loaded 2 tools from MCP server.



Graph compiled. Sending query to the agent...

=== Final Response ===
[Skipped LLM call: set OPENAI_API_KEY to run this agent for real]
